# Embedding Drift Detection in Fraud Detection Pipeline

This notebook demonstrates how to detect and visualize embedding drift in a
production fraud detection system. Embedding drift occurs when the distribution
of transaction embeddings shifts over time -- due to changing consumer behavior,
new fraud patterns, seasonal effects, or upstream model/data changes.

**Design rationale -- MMD as the sole drift metric:**

Dense embedding dimensions are highly entangled. Univariate tests (KS per
dimension) suffer from massive multiple-testing problems and miss multivariate
rotations. Cosine distance only captures mean shift. PCA explained-variance
comparisons miss mean shift entirely. Only MMD -- a kernel-based two-sample
test that operates on the full joint distribution -- correctly assesses
high-dimensional distributional divergence.

**Goals:**

1. Split the Sparkov dataset by time into reference (first 3 months) and
   production (months 4-6) periods.
2. Generate real embeddings for both periods using the local sentence-transformer
   model (all-MiniLM-L6-v2, 384 dimensions). Natural temporal drift in the data
   provides the real signal.
3. Compute MMD drift metric (the sole metric for high-dimensional embeddings).
4. Visualize drift evolution over sliding time windows.
5. Analyze per-category drift patterns via heatmap.
6. Run the EmbeddingDriftDetector for severity classification.
7. Illustrate drift types and threshold band concepts with schematic diagrams.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import SparkovDataLoader
from src.drift_detection.metrics import maximum_mean_discrepancy
from src.drift_detection.detectors import EmbeddingDriftDetector, DriftSeverity
from src.visualization.plots import (
    plot_drift_metrics_over_time,
    plot_drift_heatmap,
    plot_severity_timeline,
)
from src.visualization.schematics import (
    draw_drift_types_diagram,
    draw_threshold_bands,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

print("Libraries loaded successfully.")

In [ ]:
# Load and prepare the dataset
loader = SparkovDataLoader(data_path="../data/fraudTrain.csv")
df = loader.load()

df["trans_datetime"] = pd.to_datetime(df["trans_date_trans_time"])
df = df.sort_values("trans_datetime").reset_index(drop=True)

# Determine the time range
min_date = df["trans_datetime"].min()
max_date = df["trans_datetime"].max()
total_months = (max_date.year - min_date.year) * 12 + (max_date.month - min_date.month)
print(f"Date range: {min_date.date()} to {max_date.date()} ({total_months} months)")

# Split: first 3 months = reference, months 4-6 = production
cutoff_ref = min_date + pd.DateOffset(months=3)
cutoff_prod = min_date + pd.DateOffset(months=6)

df_ref = df[df["trans_datetime"] < cutoff_ref]
df_prod = df[(df["trans_datetime"] >= cutoff_ref) & (df["trans_datetime"] < cutoff_prod)]

print(f"Reference period: {df_ref['trans_datetime'].min().date()} to {df_ref['trans_datetime'].max().date()}")
print(f"  Transactions: {len(df_ref):,}")
print(f"Production period: {df_prod['trans_datetime'].min().date()} to {df_prod['trans_datetime'].max().date()}")
print(f"  Transactions: {len(df_prod):,}")

In [ ]:
# Generate real embeddings for reference and production periods
# using the local sentence-transformer (all-MiniLM-L6-v2, 384 dimensions).
# Falls back to random vectors if sentence-transformers is not installed.

rng = np.random.default_rng(seed=42)

# Subsample for tractability
n_ref = min(3000, len(df_ref))
n_prod = min(3000, len(df_prod))

ref_idx = rng.choice(len(df_ref), size=n_ref, replace=False)
prod_idx = np.sort(rng.choice(len(df_prod), size=n_prod, replace=False))

df_ref_sample = df_ref.iloc[ref_idx].reset_index(drop=True)
df_prod_sample = df_prod.iloc[prod_idx].reset_index(drop=True)

# Generate transaction texts
ref_texts = loader.generate_transaction_texts(df_ref_sample).tolist()
prod_texts = loader.generate_transaction_texts(df_prod_sample).tolist()

try:
    from src.embeddings.generator import LocalEmbeddingGenerator

    generator = LocalEmbeddingGenerator()
    EMBEDDING_DIM = generator.dimensions
    print(f"Using LocalEmbeddingGenerator (model: all-MiniLM-L6-v2, dim={EMBEDDING_DIM})")

    ref_embeddings = generator.encode_texts(ref_texts)
    prod_embeddings = generator.encode_texts(prod_texts)
except ImportError:
    print("WARNING: sentence-transformers is not installed. Falling back to random embeddings.")
    EMBEDDING_DIM = 384
    ref_embeddings = rng.standard_normal((n_ref, EMBEDDING_DIM)).astype(np.float32)
    prod_embeddings = rng.standard_normal((n_prod, EMBEDDING_DIM)).astype(np.float32)

# Normalize to unit length
ref_norms = np.linalg.norm(ref_embeddings, axis=1, keepdims=True)
ref_embeddings = ref_embeddings / ref_norms

prod_norms = np.linalg.norm(prod_embeddings, axis=1, keepdims=True)
prod_embeddings = prod_embeddings / prod_norms

print(f"Reference embeddings: {ref_embeddings.shape}")
print(f"Production embeddings: {prod_embeddings.shape}")

## Computing Drift with MMD

We compute MMD (Maximum Mean Discrepancy) on the full reference vs. production
comparison. MMD is the sole drift metric used because dense embedding dimensions
are highly entangled -- univariate tests miss multivariate rotations, cosine
distance only captures mean shift, and PCA-based metrics miss mean shift entirely.

| Metric | What it measures |
|--------|------------------|
| **MMD (RBF kernel)** | Full distributional divergence via kernel two-sample test |

In [ ]:
# Compute MMD drift metric
mmd_result = maximum_mean_discrepancy(ref_embeddings, prod_embeddings, kernel="rbf")

# Display result
summary_rows = [{
    "Metric": "MMD (RBF)",
    "Value": f"{mmd_result.value:.6f}",
    "p-value": f"{mmd_result.p_value:.4f}" if mmd_result.p_value is not None else "N/A",
    "Significant": mmd_result.is_significant,
}]

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## MMD Drift Over Time

To track how drift evolves, we split the production period into sliding windows
and compute MMD for each window against the fixed reference set. This reveals
whether drift is sudden or gradual.

In [ ]:
# Sliding-window drift computation (MMD only)
n_windows = 12
window_size = n_prod // n_windows

window_results = []
window_labels = []

for i in range(n_windows):
    start_idx = i * window_size
    end_idx = min(start_idx + window_size, n_prod)
    window_emb = prod_embeddings[start_idx:end_idx]

    if window_emb.shape[0] < 2:
        continue

    mmd_res = maximum_mean_discrepancy(ref_embeddings, window_emb, kernel="rbf")

    window_results.append({
        "window": i + 1,
        "mmd": mmd_res.value,
        "mmd_p_value": mmd_res.p_value,
        "mmd_significant": mmd_res.is_significant,
    })
    window_labels.append(f"W{i+1}")

window_df = pd.DataFrame(window_results)
print(window_df.to_string(index=False))

In [ ]:
# Plot MMD drift over time with threshold bands
fig = plot_drift_metrics_over_time(
    window_df=window_df,
    window_col="window",
    metric_cols=["mmd"],
    title="MMD Drift Evolution Across Production Windows",
    thresholds={
        "nominal": 0.02,
        "warning": 0.08,
        "critical": 0.20,
    },
)
plt.tight_layout()
plt.show()

## Drift by Merchant Category

Drift does not affect all transaction types uniformly. Some merchant categories
may experience more pronounced distributional shift due to seasonal patterns,
promotional events, or targeted fraud campaigns. Analyzing per-category MMD
helps prioritize monitoring and retraining efforts.

In [ ]:
# Compute per-category drift using MMD
ref_categories = df_ref_sample["category"].values
prod_categories = df_prod_sample["category"].values

unique_categories = sorted(set(ref_categories) & set(prod_categories))

# Limit to top 10 categories by transaction volume for readability
cat_counts = pd.Series(np.concatenate([ref_categories, prod_categories])).value_counts()
top_categories = cat_counts.head(10).index.tolist()

category_drift = {}
for cat in top_categories:
    ref_cat_mask = ref_categories == cat
    prod_cat_mask = prod_categories == cat

    ref_cat_emb = ref_embeddings[ref_cat_mask]
    prod_cat_emb = prod_embeddings[prod_cat_mask]

    if ref_cat_emb.shape[0] < 2 or prod_cat_emb.shape[0] < 2:
        continue

    mmd_res = maximum_mean_discrepancy(ref_cat_emb, prod_cat_emb, kernel="rbf")

    category_drift[cat] = {
        "mmd": mmd_res.value,
    }

cat_drift_df = pd.DataFrame(category_drift).T
cat_drift_df.index.name = "category"
print(cat_drift_df.to_string())

In [ ]:
# Drift heatmap: categories x metrics
fig = plot_drift_heatmap(
    drift_df=cat_drift_df,
    title="Drift Intensity by Merchant Category",
    cmap="YlOrRd",
)
plt.tight_layout()
plt.show()

## Severity Classification

The `EmbeddingDriftDetector` uses MMD as the sole drift metric and classifies
drift severity as NONE, LOW, MODERATE, HIGH, or CRITICAL based on configurable
thresholds. Because MMD operates on the full joint distribution, no ensemble
voting is needed -- a single, statistically principled metric provides the
severity signal directly.

In [ ]:
# Run the MMD-based drift detector on each time window
detector = EmbeddingDriftDetector()

# Build production windows for the detector
production_windows = []
for i in range(n_windows):
    start_idx = i * window_size
    end_idx = min(start_idx + window_size, n_prod)
    window_emb = prod_embeddings[start_idx:end_idx]
    if window_emb.shape[0] < 2:
        continue
    window_start = f"W{i+1}_start"
    window_end = f"W{i+1}_end"
    production_windows.append((window_start, window_end, window_emb))

reports = detector.evaluate_windowed(ref_embeddings, production_windows)

# Display severity timeline
severity_data = []
for idx, report in enumerate(reports):
    severity_data.append({
        "window": idx + 1,
        "overall_severity": report.overall_severity.value,
        "n_production": report.n_production,
        "recommended_actions": "; ".join(report.recommended_actions),
    })
    # Print MMD detail for each window
    mmd_result = report.metric_results[0]
    print(f"Window {idx+1}: severity={report.overall_severity.value}, MMD={mmd_result.value:.6f}, p={mmd_result.p_value:.4f}")

print()
severity_df = pd.DataFrame(severity_data)
print(severity_df[["window", "overall_severity", "n_production"]].to_string(index=False))

In [ ]:
# Plot severity timeline
fig = plot_severity_timeline(
    severity_df=severity_df,
    window_col="window",
    severity_col="overall_severity",
    title="Drift Severity Classification Over Production Windows",
)
plt.tight_layout()
plt.show()

## Drift Types Visualization

Embedding drift can manifest in four distinct patterns, each with different
operational implications:

1. **Sudden drift** -- abrupt shift, often caused by a data pipeline change or model update.
2. **Gradual drift** -- slow, steady divergence from seasonal trends or evolving fraud tactics.
3. **Incremental drift** -- step-wise changes, e.g., after periodic retraining.
4. **Recurring drift** -- cyclical patterns tied to weekends, holidays, or pay cycles.

The schematic below illustrates these four patterns.

In [ ]:
# Draw the four drift type patterns
fig = draw_drift_types_diagram(
    title="Four Common Embedding Drift Patterns",
    figsize=(14, 8),
)
plt.tight_layout()
plt.show()

## Threshold Bands

The drift detection system uses three severity zones to guide operational response:

- **Nominal (green):** Drift is within expected statistical variation. No action required.
- **Warning (yellow):** Drift exceeds baseline noise. Increase monitoring frequency;
  investigate potential root causes.
- **Critical (red):** Drift is severe enough to degrade model performance. Trigger
  model retraining, activate fallback scoring, or escalate to the fraud operations team.

The diagram below shows how a drift metric time series maps to these zones.

In [ ]:
# Draw threshold bands with MMD trajectory
fig = draw_threshold_bands(
    title="Drift Severity Threshold Bands (MMD)",
    nominal_upper=0.02,
    warning_upper=0.08,
    critical_upper=0.20,
    sample_trajectory=window_df["mmd"].values if len(window_df) > 0 else None,
    figsize=(14, 6),
)
plt.tight_layout()
plt.show()

## Key Findings and Business Implications

**Why MMD is the sole drift metric:**

Dense embedding dimensions are highly entangled. Univariate tests (KS per
dimension) suffer from massive multiple-testing problems and miss multivariate
rotations. Cosine distance only captures mean shift. PCA explained-variance
comparisons miss mean shift entirely. Only MMD -- a kernel-based two-sample
test -- correctly assesses the full high-dimensional distribution.

**Findings from this analysis:**

1. **Drift is gradual and cumulative.** The drift grows over the production
   period. Early windows show low or no drift; later windows cross into
   warning and critical territory. This mimics real-world scenarios where fraud
   tactics evolve incrementally.

2. **MMD provides a principled signal.** As a proper two-sample test with
   permutation-based p-values, MMD avoids the false positives that plague
   univariate tests on entangled embedding dimensions while remaining
   sensitive to genuine distributional shifts.

3. **Category-level heterogeneity.** Some merchant categories drift more than others.
   Monitoring MMD at the category level allows targeted retraining rather than
   expensive full-model refreshes.

4. **Threshold bands enable graduated response.** Rather than a binary
   drift/no-drift alarm, the severity system (NONE through CRITICAL) lets
   operations teams respond proportionally -- from increased monitoring through
   to emergency fallback scoring.

**Recommended operational playbook:**

| Severity | Action |
|----------|--------|
| NONE / LOW | Continue normal operations. Log for trend analysis. |
| MODERATE | Increase monitoring frequency. Begin root-cause investigation. |
| HIGH | Trigger retraining pipeline. Activate enhanced manual review. |
| CRITICAL | Engage fallback rule-based engine. Halt automated approvals. Emergency retrain. |

These findings demonstrate that continuous MMD-based embedding drift monitoring
is essential for maintaining fraud detection accuracy in production systems where
the underlying data distribution is non-stationary.